# 零、总述

语言模型处理文本时会将其分成小块，称为**词元**。为了理解自然语言。语言模型需要将词元转换为数值表示，即**嵌入向量**，如下图所示

<div align="center">
    <img src="./resources/p1.png" style="zoom:60%;" >
</div>

In [1]:
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 859.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Python >=3.8,<3.11; 1.8.0rc3 Requires-Python >=3.8,<3.11; 1.8.0rc4 Requires-Python >=3.8,<3.11; 1.8.1 Require

# 一、LLM 的分词

**词元** 不仅是模型的输出单位，也是模型查看输入的方式。发送给模型的提示词首先被分解成词元

## 1.1 下载并运行LLM

首先加载模型及其分词器，然后我们就能实际的生成了

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

声明 prompt，然后对其进行分词，再将这些词元传递给模型，模型随后生成输出。

关于如下的函数调用，说明两点：

1. 分词器中的 `return_tensors` 参数:

 `return_tensors` : str 或 TensorType

 如果被设置，将返回 tensors 而不是 Python 的整数列表，它支持一下两个值
- `pt`: 返回 PyTorch 的 `torch.Tensor` 对象
- `np`: 返回 NumPy 的 `np.ndarray` 对象

2. model.generate 函数

🅰️ 输入参数

① **inputs**: 依赖于 modlity 的 `torch.Tensor` 对象，可选

它是一个序列，生成时作为提示词使用，编码时作为模型的输入。如果为 `None`，此方法使用 `bos_token_id` 和 批量大小为 1 来初始化。对于仅解码器的模型， 此参数应该是 `input_ids` 的格式，对于编码器-解码器模型，此参数可以是 `input_ids`, `input_values`, `input_features` 或 `pixel_values`

② **generation_config**: GenerationConfig 对象, 可选

 用作本次生成调用的基础参数配置，传递给 `generate` 方法的 `**kwargs` 中，如果有参数名与 `generation_config` 中的属性匹配，则会覆盖 `generation_config` 中对应的设置。如果没有提供 `generation_config`, 则会使用默认配置，其默认配置的优先级如下：

 a. 如果模型中 `generation_config.json` 文件存在的话，从此文件中读取

 b. 从模型的配置中加载

 请注意, 未指定的参数值会继承自 `GenerationConfig` 中的默认值，因此，在配置生成参数时，应查看 `GenerationConfig` 的相关文档

③ **logits_processor**: LogitsProcessorList 的对象，可选

自定义的 logits 处理器的列表，用于补充根据参数和生成配置自动构建的默认 logits 处理器。如果传入的某个 logits 处理器已经通过参数或 `generation_config` 创建过，那么会抛出异常。此功能主要面向高级用户

[补充] `logits_processor` 可以理解为：在模型的每一步生成下一个 token 前，对候选 token 的分数做一次 “规则处理”

④ **stopping_criteria**: StoppingCriterialList 的对象，可选

自定义的 stopping criteria 的列表，用于补充根据参数和生成配置自动构建的默认 stopping criteria。如果传入的某个 stopping criteria 已经通过参数和 `generation_config` 创建过，那么会抛出异常。如果你的 stopping criteria 依赖 `scores` 输入，一定要请确保你传递了 `return_dict_in_generate=True`, `output_scores=True`. 此功能主要面向高级用户

[补充] `stopping_criteria`：停止的标准，可以理解为：模型什么时候停止生成下一个 token

⑤ **prefix_allowed_tokens_fn**: `Callable[[int, torch.Tensor], list[int]]`， 可选

如果提供，此函数会限制在每一步允许束搜索的 token 数。如果没提供，则无限制。此函数接收两个参数：`batch_id` 和 `input_ids`。它必须返回一个列表，其中包含模型在下一次生成步骤中允许的 token，此列表基于 `batch_id` 和 之前生成的`input_ids` 来确定。

⑥ **synced_gpus**: `bool`， 默认 `False`, 可选

决定是否同步所有 GPU 的生成循环，也可以说：当循环生成的 token 长度到达了设置的 `max_length` 时是否继续生成。除非显示设置该值，否则当使用多个 GPU，并且启用了 `FullyShardedDataParallel` 或 `DeepSpeed ZeRO Stage 3` 时，该标志会被设置为 `True`, 以避免某个 GPU 比其他 GPU 更早完成生成而导致死锁。否则，默认值为 `False`。

⑦ **assistant_model**: PreTrainedModel 的对象, 可选

加速模型生成的辅助模型。这个辅助的模型必须和模型有完全相同的分词器。这个模型应当更小

⑧ **streamer**: BaseStreamer 的对象，可选

用于流式生成序列的 Streamer 对象。生成的 token 通过 `streamer.put(token_ids)` 传递，且流式生成器 streamer 负责进一步的生成

⑨ **negative_prompt_ids**: 形状为 `(batch_size, sequence_length)` 的 `torch.LongTensor`，可选

此参数为负向提示词的 token_ids, 某些 processor(logits processor) ，比如 CFG processor, 需要它来控制模型原理负向提示词描述的方向。其中，
- processor 也叫做 logits processor，是生成过程中负责修改 token 分数的组件。
- CFG：Classifier-Free Guidance, 无分类器引导，一种*生成控制技术*，核心的作用是：
> 让模型生成结果更靠近你想要的条件，同时远离你不想要的方向

其核心公式如下：
$$
guided\_logits = negative\_logits + scale \times (positive\_logits - negative\_logits)
$$
其中，`scale` 是无分类器引导的超参数

⑩ **negative_prompt_attention_mask**: 形状为 `(batch_size, sequence_length)` 的 `torch.LongTensor`，可选

`negative_prompt_ids` 的注意力掩码


➡️ 返回值: `ModelOutput` 对象 或 `torch.LongTensor`

如果 `return_dict_in_generate=True` 或 `config.return_dict_in_generate=True`,则返回 ModelOutput，否则返回 `torch.LongTensor`

如果模型**不是**编码器-解码器模型(即`model.config.is_encoder_decoder=False`), 可能的 `ModelOutput` 的类型为：
- `GenerateDecoderOnlyOutput`
- `GenerateBeamDecoderOnlyOutput`

如果模型**是**编码器-解码器模型(即`model.config.is_encoder_decoder=True`)，可能的 `ModelOutput` 的类型为：
- `GenerateEncoderDecoderOutput`
- `GenerateBeamEncoderDecoderOutput`


In [3]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# 将 prompt 分词化
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# 生成文本
generate_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=20
)

print(generate_output)

print(tokenizer.decode(generate_output[0]))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')
Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


从代码中可以看到，模型实际上并**没有直接处理**提示词文本。相反，输入提示词是由 tokenizer 处理的。tokenizer 在变量 input_ids 中返回了模型所需的信息，随后模型将其用作输入。

让我们打印 input_ids 看看它包含什么：

In [4]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


这就是 LLM 输出的响应，如上述代码运行结果所示。每个整数都是特定词元（字符、词或词的一部分）的唯一ID。这些ID是分词器内部的一张词元表的索引，该表包含了分词器能够识别的所有词元。

如果想查看这些 ID 对应的词元，可以使用 tokenizer 的 decode 方法将 ID 转换成人类可阅读的文本

In [5]:
for id in input_ids[0]:
    print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


这就是分词器分解输入提示词的过程。

需要注意以下几点：
- 第一个词元是ID 1(`<s>`),这是一个表示文本开始的特殊词元
- 一些词元是完整的单词（例如 Write、an、email等）
- 一些词元史单词的部分（例如 apolog、izing、trag、ic等）
- 标点符号是独立的词元

**注意**：

空格字符不用单独的词元表示，代表词的一部分的 token (如izing 和 ic) 在开头有一个特殊的隐藏字符，表示它们与文本中前面的词元相连。没有这个特殊字符的词元前面则都被视为有一个空格

在输出端，我们可以通过打印 `generation_output` 变量来查看模型生成的词元，打印出的内容同时包括输入和输出词元

In [6]:
print(generate_output)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')


在上述例子中，模型生成了词元3323(Sub)，接着是词元622(ject)。它们一起组成了Subject这个词。然后是词元29901，即冒号，等等。就像输入端一样，我们需要分词器在输出端将词元ID转换为实际文本。我们使用分词器的decode方法来实现这一点。我们可以传入单个词元ID或它们的列表：

In [7]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


## 1.2 比较不同的分词器

### 1.2.1 分词器如何分解文本

决定分词器如何分解输入提示词的因素主要有三个:

**首先**，在模型设计时，模型创建者会选择一种**分词方法**。流行的方法包括字节对编码（BPE，byte pair encoding，广泛用于GPT模型）和WordPiece（用于BERT模型）。这些方法的相似之处在于，它们都旨在找到一组尽可能高效的词元来表示文本数据集，但它们采用不同的方式实现这一目标。

**其次**，在选择方法之后，我们需要做出一些**分词器设计选择**，如*词表大小*和*使用哪些特殊词元*。

**最后**，分词器需要在**特定数据集**上进行训练，以建立能最好地表示该数据集的词表。即使我们设置相同的方法和参数，在英语文本数据集上训练的分词器也会与在代码数据集或多语言文本数据集上训练的分词器不同。

### 1.2.2 词级、子词级、字符级和字节级分词

我们刚才讨论的分词方案被称为子词级分词(subword tokenization)。这是最常用的分词方案，但并非唯一的方案。图2-6展示了四种主要的分词方式。

1. **词级分词**


这种方法在早期的 word2vec 等模型中很常见，但在 NLP 中的使用越来越少。不过，由于其实用性，它在推荐系统等 NLP 以外的场景中得以应用，**词级分词的一个挑战是: 分词器可能无法处理分词器训练完成之后才出现在数据集中的新词**。这也导致词表中存在大量仅有细微差别的词元（如apology、apologize、apologetic、apologist）。这个问题可以通过*子词级分词*来解决，因为它有一个 apolog 词元，还有后缀词元（如-y、-ize、-etic、-ist），这些后缀词元与许多其他词元共用，从而形成更具表达能力的词表。


2. **子词级分词**

这种方法包含词和词的一部分。除了前面提到的词表更具表达能力外，这种分词方法的另一个**优势是可以表示新词**，因为它可以将新词元分解成较小的字符，这些字符通常都在词表中。

3. **字符级分词**

它仅仅依赖最原始的字母。虽然这使得分词更容易，但是建模却变得困难。在*子词级分词*中，模型可以用一个词元表示 `play`, 而使用字符级分词的模型则需要建模拼写出 p-l-a-y 的信息，此外还要建模序列的其他部分。相比于*字符级分词*，*子词级分词*可以在 Transformer 模型有限的上下文长度内，容纳更多文本。因此，在上下文长度为 1024 的模型中，使用*子词级分词*可以容纳*字符级分词*约三倍的文本。

4. **字节级分词**

这种方法是将词元分解为表示 Unicode 字符的单个字节。它们也被称为“免分词编码”，研究结果表明，*字节级分词*是一种颇具竞争力的方法，尤其在多语言的场景中。

这里需强调一个区别：某些*子词级分词器*也会在其此表中将字节作为词元，在遇到无法用其他方式表示的字符时，这是最终的备选方案，例如 GPT-2 和 RoBERTa 分词器就是这样做的。


下面的表格展示了上述四种分词方法

<div align="center">

<table style="font-size: 20px; text-align: center; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="padding: 10px 18px; text-align: center;">粒度</th>
      <th style="padding: 10px 18px; text-align: center;">表示结果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 10px 18px;">原始文本</td>
      <td style="padding: 10px 18px;">Have the 🎵 bards who precede...</td>
    </tr>
    <tr>
      <td style="padding: 10px 18px;">词级词元</td>
      <td style="padding: 10px 18px;">
        <code>Have</code> <code>the</code> <code>🎵</code> <code>bards</code> <code>who</code> <code>preceded</code> <code>...</code>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px 18px;">子词级词元</td>
      <td style="padding: 10px 18px;">
        <code>Have</code> <code>the</code> <code>🎵</code> <code>bard</code> <code>s</code> <code>who</code> <code>preced</code> <code>ed</code> <code>...</code>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px 18px;">字符级词元</td>
      <td style="padding: 10px 18px;">
        <code>H</code> <code>a</code> <code>v</code> <code>e</code> <code>&lt;space&gt;</code>
        <code>t</code> <code>h</code> <code>e</code> <code>&lt;space&gt;</code>
        <code>🎵</code> <code>&lt;space&gt;</code>
        <code>b</code> <code>a</code> <code>r</code> <code>d</code> <code>s</code>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px 18px;">字节级词元</td>
      <td style="padding: 10px 18px;">每个字符拆成 UTF-8 编码的格式</td>
    </tr>
  </tbody>
</table>

</div>

### 1.2.3 比较训练好的 LLM分词器

分词器中出现的词元是由三个主要因素决定的：
- 分词方法
- 用于初始化分词器的参数和特殊词元
- 用于训练分词器的数据集

让我们使用一个工具函数来指定不同的分词器对文本进行分词，并用彩色背景显示每个词元，函数定义如下：

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) + '\x1b[0m',
            end=' '
        )

我们以下面的 `text` 为例，可以看到，此文本中包含：
- 大小写
- 英语之外的语言
- emoji
- 代码（包括关键字和经常用于缩进的空白字符）
- 数字

In [9]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

下面，我们看看不同的分词器是如何对此段文本进行分词的

#### 1.2.3.1 BERT基座模型（大小写敏感）(2018)

分词方法：WordPiece

词表大小：28996

特殊词元：`[UNK]`、`[SEP]`、`[PAD]`、`[CLS]`、`[MASK]`
- `[UNK]`: 未知词元
- `[SEP]`: 分隔符词元，用于支持需要为模型提供两端文本的特定任务，在此情况下，模型被称为 cross-encoder
- `[PAD]`: 填充词元
- `[CLS]`: 分类词元，主要用于分类任务的特殊词元
- `[MASK]`: 掩码词元，在训练过程中用于隐藏词元


In [10]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

对于此模型的分词，我们注意到如下特点：
- CAPITALIZATION 被表示为 8 个词元
- ***此分词器会在输入文本前后添加一个起始词元 `[CLS]` 和结束词元 `[SEP]`***

#### 1.2.3.2 BERT基座模型（大小写不敏感）(2018)

分词方法：WordPiece

词表大小：30552

特殊词元：`[UNK]`、`[SEP]`、`[PAD]`、`[CLS]`、`[MASK]`
- `[UNK]`: 未知词元
- `[SEP]`: 分隔符词元，用于支持需要为模型提供两端文本的特定任务，在此情况下，模型被称为 cross-encoder
- `[PAD]`: 填充词元
- `[CLS]`: 分类词元，主要用于分类任务的特殊词元
- `[MASK]`: 掩码词元，在训练过程中用于隐藏词元


In [11]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

对于此模型的分词，我们注意到如下特点：
- **换行符消失**，这使得模型无法识别通过换行符体现的信息
- 所有文本**都变成小写**
- capitalization 这个词(原文为 CAPITALIZATION)被编码为两个子词元： `capital` 和 `##ization`。需要说明的是：`##` 符号用来表示这个词元是与前面的词元相连的部分词元。分词中还约定没有 `##` 前缀的词元前面应该有一个空格
- **emoji 和中文字符消失了**，被替换成了 `[UNK]`
- ***此分词器会在输入文本前后添加一个起始词元 `[CLS]` 和结束词元 `[SEP]`***

#### 1.2.3.3 GPT-2(2019)

分词方法：BPE(Byte Pair Encoding)

词表大小：50257

特殊词元：`<|endoftext|>`, 它通常用于表示：一段文本结束、文档结束、样本分隔

In [12]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

使用 GPT-2 分词器，我们注意到以下特点：
- 分词器**保留了换行符**
- 保留了大小写，CAPITALIZATION 被表示为四个词元
- emoji 和 中文字符，分别被表示为多个词元，虽然显示为 `?` 字符，但它们实际上代表不同的词元。我们可以通过打印 `tokenizer.decode([8532, 236, 113])` 来验证，它会输出 🎵
- 两个制表符被表示为两个词元（词元号为 197），三个制表符被表示为三个词元（词元号为 220）

In [15]:
gpt_2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

print("GPT2_Tokenizer 分词后的IDs为：")
print(gpt_2_tokenizer(text, return_tensors="pt").input_ids.to("cuda"))
print("GPT2_Tokenizer 对 [8532, 236, 113] 解码")
print(gpt_2_tokenizer.decode([8532, 236, 113]))

GPT2_Tokenizer 分词后的IDs为：
tensor([[  198, 15823,   290, 20176, 40579, 14887,  6234,   198,  8582,   236,
           113, 16268,   116,   253,   198, 12860,    62,    83,   482,   641,
         10352,  6045,  1288,   361,  6624, 18189,  2073,    25,   734, 22524,
         11097,   220,   220,   220,   366,  7683, 22524,    25,   366,   220,
           220,   220,   220,   220,   220,   366,   198,  1065,    13,    15,
             9,  1120,    28,  8054,   198]], device='cuda:0')
GPT2_Tokenizer 对 [8532, 236, 113] 解码
 Dog��


#### 1.2.3.4 FLAN-T5(2022)

分词方法：SentencePiece

词表大小：32100

特殊词元：`<unk>`、`<pad>`

In [16]:
show_tokens(text, "google/flan-t5-small")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600 </s> 

使用此模型分词后，我们注意到以下两点：
- 该分词方法中**没有换行符和空白字符的词元表示**，这会使模型处理代码变得具有挑战性
- emoji 和 中文字符都被替换为 `<unk>`

#### 1.2.3.5 GPT-4(2023)

分词方法：BPE(Byte Pair Encoding)

词表大小：略多于10,0000

特殊词元：

<div style="display: flex; justify-content: center; margin: 24px 0;">
  <table style="border-collapse: collapse; font-size: 16px; text-align: center;">
    <thead>
      <tr>
        <th style="border: 1px solid #ccc; padding: 10px 18px;">名称</th>
        <th style="border: 1px solid #ccc; padding: 10px 18px;">token id</th>
        <th style="border: 1px solid #ccc; padding: 10px 18px;">含义</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;|endoftext|&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>100257</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">文本结束标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;|fim_prefix|&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>100258</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 前缀标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;|fim_middle|&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>100259</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 中间标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;|fim_suffix|&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>100260</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 后缀标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;|endofprompt|&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>100276</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">prompt 结束标记</td>
      </tr>
    </tbody>
  </table>
</div>

我们来说一下 **FIM**，也就是上表中的 **Fill-in-the-middle** 的缩写，可以将其理解为补全中间内容，也就是说，模型不仅仅是从左到右续写，而是可以根据前文和后文，补全中间缺失的部分。它最常见于 **代码补全** 场景

👉🏻举个例子

原始代码：
```python
def greet(name):
    # todos: 此处需补全
    ...

print(greet("Tom"))
```

现在要补全 `#todos：` 的部分，可以将其分词（构造）成如下形式：

```text
<|fim_prefix|>
def greet(name):

<|fim_suffix|>
print(greet("Tom"))

<|fim_middle|>
```

这样的话，模型就可能生成：
```text
return f"Hello, {name}"
```

In [17]:
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.01M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.23M [00:00<?, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 12 . 0 * 50 = 600 
 

GPT-4 的分词器行为与 GPT-2 类似，一些区别如下：
- GPT-4 分词器**将四个空白字符表示为单个词元**。实际上，它甚至为各种长度的空白字符序列都设有特定的词元
- Python 关键字 `elif` 在 GPT-4 中有自己的词元
- GPT-4 分词器使用更少的词元来表示大多数词，例如 CAPITALIZATION(当前被表示为两个词元，在 GPT-2 中是四个)，另外还有 `tokens` (当前的一个词元代替了之前的三个)

#### 1.2.3.6 StarCoder2(2024)


StarCoder2 是一个专注于生成代码的150亿参数模型

分词方法：BPE(Byte Pair Encoding)

词表大小：49152

特殊词元（示例）：

<div style="display: flex; justify-content: center; margin: 24px 0;">
  <table style="border-collapse: collapse; font-size: 16px; text-align: center;">
    <thead>
      <tr>
        <th style="border: 1px solid #ccc; padding: 10px 18px;">名称</th>
        <th style="border: 1px solid #ccc; padding: 10px 18px;">类型</th>
        <th style="border: 1px solid #ccc; padding: 10px 18px;">含义</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;filename&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">元数据词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">文件名 / 文件路径标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;reponame&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">元数据词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">代码所属仓库名称标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;gh_stars&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">元数据词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">GitHub 仓库 Star 数量标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;|endoftext|&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">特殊词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">文本结束标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;fim_prefix&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">中间填充词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 前缀标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;fim_middle&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">中间填充词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 中间生成位置标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;fim_suffix&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">中间填充词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 后缀标记</td>
      </tr>
      <tr>
        <td style="border: 1px solid #ccc; padding: 10px 18px;"><code>&lt;fim_pad&gt;</code></td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">中间填充词元</td>
        <td style="border: 1px solid #ccc; padding: 10px 18px;">Fill-in-the-middle 填充标记</td>
      </tr>
    </tbody>
  </table>
</div>

In [18]:
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

此模型的分词器，我们注意到如下特点：
- 类似于 GPT-4，它将一系列的空白字符编码为单个词元
- 此方法的主要不同之处是 **它给每个数字都分配了词元**, 因此，会看到 600 变为 `6`,`0`,`0`。这种设计可以更好的表示数字和数学的概念。而反观其他模型，它可能把 871 分为 8 和 71，或者 87 和 1，这种的表示方式可能会让模型混淆对数字的理解

#### 1.2.3.7 Galactica

Galactica 模型专注于科学知识领域，是在大量的科学论文、参考资料和知识库上训练而来的。它特别注重分词方式，旨在更好地理解其所表示数据集的细微差别。例如，它包含了用于表示引用、推理、数学、氨基酸序列和 DNA 序列的特殊词元

分词方法：BPE(Byte Pair Encoding)

词表大小：5,0000

特殊词元：
- 常见词元：`<s>`、`<pad>`、`<unk>`
- (论文)引用相关：`[START_REF]`、`[END_REF]`
- 逐步推理词元：`<work>`，模型用它来进行思维链(Chain-of-Thought, CoT)推理

In [19]:
show_tokens(text, "facebook/galactica-1.3b")

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]


 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "      "  Three  t abs :   "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

Galactica 分词器的分词，我们注意如下特点：
- 将不同长度的空白字符序列分别编码为单个词元
- **它对不同长度的制表符 `\t` 也分配成了单个词元**

#### 1.2.3.8 Phi-3（和 Llama 2）

Phi-3 重用了 Llama 2 的分词器

分词方法：BPE(Byte Pair Encoding)

词表大小：32000

特殊词元：
- 常规词元：`<|endoftext|>`
- 对话词元：`<|user|>`、`<|assistant|>`、`<|system|>`

In [20]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

最终，有三大类设计上的选择决定了分词器如何分解文本：
- **<div style="color: red">分词方法</div>** 目前最流行的是 BPE 算法
- **<div style="color: red">初始化分词器的参数</div>** 词表大小、特殊词元、大小写处理策略等等
- **<div style="color: red">训练分词器的目标数据所在的领域</div>** 不同的数据集


总览如下：

<center>
<img src="./resources/compare_llm_tokenizer.png">
</center>

# 二、词元嵌入(Word Embedding From LLMs)

模型中的 token 表示可以分为两个阶段：

1. 通过词元嵌入矩阵将词元编号转换成初始向量
2. 模型**结合上下文**，对这些初始的向量进行加工，得到更准确的上下文语义表示

这些经过语言模型加工后的向量不仅可以用于继续预测下一个词元，也可以作为其他任务的输入，例如 NER（Named-Entity Recognition，命名实体识别）和抽取式文本摘要等。进一步地，DALL·E、Midjourney、Stable Diffusion 等图像生成系统也会使用类似的文本表示方式

- **命名实体识别（NER）**：从文本中识别人名、地名、组织名、时间和金额等等实体
- **抽取式文本摘要**：它是从原文中挑选最重要的句子或片段组成摘要，不重新生成新的文本，优点是忠实原文
- **Stable Diffusion**: 文生图模型，它的核心不是“直接画图”，而是 *从一团随机噪声开始，在文本提示词的引导下，一步一步去噪，最后得到清晰图像*

我们使用 DeBERTaV3 模型来做示例：

In [21]:
from transformers import AutoModel, AutoTokenizer
# 加载分词器和语言模型
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# 分词
tokens = tokenizer("Hello world", return_tensors="pt")

output = model(**tokens)[0]

print(tokens)

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

{'input_ids': tensor([[    1, 31414,   232,     2]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [22]:
print(output)

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)


In [23]:
print(output.shape)

torch.Size([1, 4, 384])


可以看到，上面 `output.shape` 的形状为 `[1, 4, 384]`，而对于 PyTorch 的 Tensor 来说，第 1 个维度是 batch 的维度，所以，此分词器是将输入的句子拆分成了 4 个词元，随后模型将这 4 个词元嵌入到了 384 维的空间中

输入的 4 个词元如下所示：

In [24]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


# 三、文本嵌入(Text Embedding: For Sentences and Whole Documents)

有一些 LLM 应用也可以处理完整的句子、段落甚至文档，它们可以生成文本的嵌入，即：用单个向量来表示长度超过一个词元的文本片段。生成文本嵌入有很多种方法，最常见的一种方法是对模型生成的所有词元嵌入的值**取平均值**。我们使用 `sentence-transformers` 这个库来举例，代码如下：

In [26]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
vector = model.encode("Best movie ever!")

print(vector.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(768,)


# 四、LLM 之外的词嵌入(Word Embeddings Beyond LLMs)

嵌入(Embedding) 不仅在文本和语言生成方面有用，另外在推荐引擎和机器人技术等领域也有作用。让我们以 word2vec 为例（我们使用 Gensim 库下载）

In [28]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.7 MB/s eta 0:00:00


In [30]:
import gensim.downloader as loader
word2vec_model = loader.load("word2vec-google-news-300")
# glove_model = loader.load("glove-wiki-gigaword-50")

In [44]:
king_vec = word2vec_model['king']

print("=============== 打印 king 的嵌入向量表示及其形状 ======================")
print("向量的前 20 项为：", king_vec[:20])
print("形状为：", king_vec.shape)
print("=============== 打印与 king 的嵌入向量表示最近的 10 个向量 ======================")
top_10 = word2vec_model.most_similar([king_vec], topn=10)
print(f"{'序号':<6}{'word':<20}{'余弦相似度':>12}")
print("-" * 70)
for idx, (word, score) in enumerate(top_10, start=1):
    print(f"{idx:<6}{word:<20}{score:>12.6f}")
print("=============== 打印 king 与 queen 的相似度(cosine similarity) ======================")
print(word2vec_model.similarity("king", "queen"))
print("=============== 打印 king 与 banana 的相似度(cosine similarity) ======================")
print(word2vec_model.similarity("king", "banana"))

=============== 打印 king 的嵌入向量表示及其形状 ======================
向量的前 20 项为： [ 0.12597656  0.02978516  0.00860596  0.13964844 -0.02563477 -0.03613281
  0.11181641 -0.19824219  0.05126953  0.36328125 -0.2421875  -0.30273438
 -0.17773438 -0.02490234 -0.16796875 -0.16992188  0.03466797  0.00521851
  0.04638672  0.12890625]
形状为： (300,)
=============== 打印与 king 的嵌入向量表示最近的 10 个向量 ======================
序号    word                       余弦相似度
----------------------------------------------------------------------
1     king                    1.000000
2     kings                   0.713804
3     queen                   0.651096
4     monarch                 0.641319
5     crown_prince            0.620422
6     prince                  0.615999
7     sultan                  0.586482
8     ruler                   0.579757
9     princes                 0.564655
10    Prince_Paras            0.543294
=============== 打印 king 与 queen 的相似度(cosine similarity) ======================
0.6510956
=============== 打

下面我们介绍一下 Word2Vec 算法

Word2Vec 的本质是：**用一个很简单的预测任务，让模型从大量文本中自动学习词的向量表示；CBOW 用上下文预测中心词，Skip-gram 用中心词预测上下文。通过去掉复杂隐藏层、使用层次 Softmax 等方法。它可以在大规模语料上高效训练处能表达语义关系的词向量**

Word2Vec 的核心目标为：把每个词表示成一个连续向量，让语义相近、上下文相似的词，在向量空间中也更接近

工程化的 Word2Vec，一般指的是：
```text
CBOW/Skip-gram + Hierarchical Softmax 或 Negative Sampling
```

下面分别介绍一下这四种结构（算法）：

1. CBOW（Continuous Bag-of-Words）

根据上下文预测中心词，它的**输入**是 context(surrounding word)，**输出**的是 current word

例如句子：
```text
I like drinking coffee in the morning
```
若窗口大小为 2，要预测 `coffee`，则输入是 `drinking` 和 `in`

---

2. Skip-gram：根据中心词预测上下文

它与 CBOW 结构正好相反，**输入**是 current word, **输出**是 surrounding words

例如句子：
```text
I like drinking coffee in the morning
```
若 current word 是 coffee, 则输入是 `coffee`，输出（预测）是 `drinking`、`in`、`like` 等等

下图展示了它们两者的区别：

<img src="./resources/word2vec1.webp" style="zoom: 60%">


---
3. Hierarchical Softmax

它的核心思想是：不在整个词表中选择一个最适合的词，而是把词表组织成一棵 Huffman 树，通过一连串二分类决策走到目标词

普通的 softmax 是一次性从整个词表中选择目标词，Hierarchical Softmax 则是把整个词表组织成一棵二叉树，**将原本的大规模多分类问题拆解成多个二分类问题**，算法的时间复杂度也从 $O(V)$ 变成了 $O(logV)$, 其中 V 是词表的大小

下面来讲解一下哈夫曼树（Huffman Tree）

Huffman Tree，是满足 **WPL（带权路径长度） 最小**的一个二叉树，它的核心思想是：权重越大的节点，离根节点越近。权重越小的节点，离根节点越远。而在 Word2Vec 中，词的权重通常是*词频（Term Frequency）*，所以，高频词的路径短，低频词路径长。所以最后会构建出一棵类似于如下这样的树：
```text
              root
             /    \
          the     node1
                 /     \
               of      node2
                      /     \
                  coffee    node3
                            /    \
                         apple  mitochondria
```

WPL(Weighted Path Length) 定义为：
$$
WPL = \sum_i weight_i \times path\_length_i
$$
其中：
- $weight_i$ 代表第 $i$ 个节点的权重
- $path\_length_i$ 代表从根节点到该节点的路径长度

---
4. Negative Sampling(负采样)

它和 Hierarchical Softmax 类似，也是 Word2Vec 里为了加速训练的一种方法。也就是说，原来模型要在整个词表中预测正确词，现在负采样改为**只判断“真实词对”和少量“假词对”**，也叫做正样本和负样本。


假设我们用 Skip-gram 训练 Word2Vec, 句子是
```text
I like drinking coffe in the morning
```
假定 current word 是 coffee,则训练的样本可以是
```text
coffee -> drinking
coffee -> in
coffee -> like
...
```
那如果词表有 100 万个词，普通的 Softmax 每训练一次，都要计算一遍，这样的话非常慢。所以，我们引入了**正样本**和**负样本**的策略，**正样本**是真实出现过的词对，**负样本**是随机抽出来的错误词对

比如，对如上的句子来说

正样本例如：
```text
(coffee, drinking) -> 1
(coffee, in) -> 1
(coffee, like) -> 1
```

负样本例如：
```text
(coffee, banana) -> 0
(coffee, car) -> 0
(coffee, government) -> 0
```
所以，现在模型的训练指标变成了**二分类交叉熵(Binary Cross-Entropy)**


# 五、推荐系统中的嵌入

在本节中，我们将使用 Word2Vec 算法，利用人工创建的音乐播放列表来嵌入歌曲。如果我们把每首歌曲都当做一个词或词元来处理，把每个播放列表当作一个句子，这些嵌入就可以用来推荐经常出现在同一个播放列表中的歌曲

首先加载包含歌曲播放列表的数据集，以及每首歌曲的元数据

In [45]:
import pandas as pd
from urllib import request

# 获取 Playlist 数据集
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# 从第 2 行之后开始逐行加载，因为前两行是元数据
lines = data.read().decode("utf-8").split('\n')[2:]

# 移除 Playlist 中的单曲
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# 加载歌曲的原数据
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [46]:
songs_df.shape

(75263, 2)

In [53]:
songs_df[:20]

,title,artist
id,,
0,Gucci Time (w\/ Swizz Beatz),Gucci Mane
1,Aston Martin Music (w\/ Drake & Chrisette Mich...,Rick Ross
2,Get Back Up (w\/ Chris Brown),T.I.
3,Hot Toddy (w\/ Jay-Z & Ester Dean),Usher
4,Whip My Hair,Willow
5,Down On Me (w\/ 50 Cent),Jeremih
6,Black And Yellow,Wiz Khalifa
7,Blowing Me Kisses,Soulja Boy
8,Lay It Down,Lloyd


让我们检查一下 Playlist

In [47]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

让我们训练 Word2Vec 这个模型

In [48]:
from gensim.models import Word2Vec
model = Word2Vec(
    playlists,
    vector_size=32,     # 嵌入的维度
    window=20,
    negative=50,
    min_count=1,       # 只要向量出现过一次，就会被加入词表
    workers=4
)

让我们以 《Fade To Black》 歌曲为例，它的歌曲ID 为 2172，它偏向于摇滚/重金属体系

In [50]:
songs_df.iloc[2172]

,2172
title,Fade To Black
artist,Metallica


可以看到： 2172 对应的歌曲名为《渐入黑暗》，其歌手是 Metallica 金属乐队

接下来，我们打印在嵌入空间中与它最相近的 10 首歌曲

In [60]:
import pandas as pd

song_id = 2172

top_10 = model.wv.most_similar(
    positive=str(song_id),
    topn=10
)

result = []

for rank, (similar_song_id, score) in enumerate(top_10, start=1):
    similar_song_id = int(similar_song_id)

    result.append({
        "rank": rank,
        "Song Id": similar_song_id,
        "title": songs_df.iloc[similar_song_id]["title"],
        "artist": songs_df.iloc[similar_song_id]["artist"],
        "Cosine similarity": round(score, 6)
    })

result_df = pd.DataFrame(result).set_index("rank")

result_df

,Song Id,title,artist,Cosine similarity
rank,,,,
1,3167,Unchained,Van Halen,0.997802
2,2849,Run To The Hills,Iron Maiden,0.997054
3,11473,Little Guitars,Van Halen,0.997049
4,5586,The Last In Line,Dio,0.996911
5,1922,One,Metallica,0.996649
6,6641,Shout At The Devil,Motley Crue,0.996427
7,2976,I Don't Know,Ozzy Osbourne,0.995944
8,5634,Mr. Brownstone,Guns N' Roses,0.995489
9,11517,Mary Had A Little Lamb,Stevie Ray Vaughan & Double Trouble,0.995377


翻译如下：

| 排名 | 英文歌名                   | 中文译名         | 歌手                                                   |
| -: | ---------------------- | ------------ | ---------------------------------------------------- |
|  1 | Unchained              | 挣脱束缚 / 无拘无束  | Van Halen（范·海伦）                                      |
|  2 | Run To The Hills       | 逃向山丘         | Iron Maiden（铁娘子乐队）                                   |
|  3 | Little Guitars         | 小吉他          | Van Halen（范·海伦）                                      |
|  4 | The Last In Line       | 队列中的最后一人     | Dio（迪奥）                                              |
|  5 | One                    | 一 / 唯一       | Metallica（金属乐队）                                      |
|  6 | Shout At The Devil     | 对着魔鬼呐喊       | Motley Crue（克鲁小丑 / 莫特利·克鲁）                           |
|  7 | I Don't Know           | 我不知道         | Ozzy Osbourne（奥兹·奥斯本）                                |
|  8 | Mr. Brownstone         | 布朗斯通先生       | Guns N' Roses（枪炮与玫瑰）                                 |
|  9 | Mary Had A Little Lamb | 玛丽有只小羊羔      | Stevie Ray Vaughan & Double Trouble（史蒂夫·雷·沃恩与双重麻烦乐队） |
| 10 | Youth Gone Wild        | 狂野青春 / 失控的青春 | Skid Row（贫民窟乐队）                                      |


可以看到，这几首歌曲也是偏向于摇滚/重金属风格

# 六、小结

<span style="color: red">分词器设计中有三个主要决策点：</span>
- **分词器算法**（如BPE，WordPiece,SentencePiece），
- **分词参数**(包括词表大小、特殊词元、大小写处理策略和不同语言的处理)
- 用于训练分词器的**数据集**

在此，我们只介绍不同的分词器算法

1. BPE（Byte Pair Encoding）
它的核心思想是：**从字符开始，不断合并语料中最常一起出现的相邻符号对**，也就是说，BPE 一开始把单词看成字符序列，然后反复统计哪两个相邻字符/子词最常一起出现，然后将它们合并成一个新的 token

2. WordPiece
它的核心思想是：**从最小单元开始，逐渐构建子词词表**，和 BPE 不同的是，WordPiece 不只是看“出现频率最高的相邻组合”，而是更关注：合并之后，是否能更好地提升语言模型的概率。另外，WordPiece 的标记词的方式有变化，它常用 `##` 表示这个 token 是接在前一个 token 后面的子词

例如：`unaffable` 可能被拆分为 `un` + `##aff` + `##able`

3. SentencePiece
SentencePiece 不是单一分词算法，而是一个分词工具，它支持 `BPE` 和 `Unigram Language Model` 算法，它最大的特点是**将输入文本当作普通字符序列处理，不依赖空格预分词**，这对中文，日文，韩文等语言特别有用，因为这些语言不是按照空格开拆分具体的词。另外，SentencePiece 通常会用特殊符号 `_` 表示空格或词的开始。

例如：`I love NLP`，可能被编码为 `_I_love_NLP`

最后，来说一下 Unigram Language Model 算法，它是**先准备一个比较大的候选子词词表，然后逐渐删除不重要的子词，留下最优词表**
